# A-SNN research restart — Evidence-1 first
Prepared 2026-10-13. Open in **Google Colab** or Jupyter and run top to bottom.
Default is an inexpensive **synthetic smoke test**, not SHD/SSC evidence.
For the registered study, select a **GPU** runtime, set `PROFILE='confirmation'`,
`CONFIRM_FULL_RUN=True`, and preferably `USE_DRIVE=True` below, then Run all.
Download, checksum verification, training, resumption, plots and ZIP export are included.

The confirmation is 80 fits (2 datasets × 10 seeds × 4 models), 15 epochs each,
plus frozen-checkpoint state ablations and compiler checks. It can outlast a
Colab session. Epoch checkpoints persist if Drive is enabled; otherwise download
the output folder before disconnecting. No TPU path or silent CPU confirmation.
Allow at least 20 GB free disk and preferably 16+ GB host RAM for official audio;
these are conservative planning allowances, not measured resource requirements.

FiTS, timing-only data and whole-fly reproduction are separately opt-in.
They do not alter the registered Evidence-1 protocol or reopen closed mechanisms.
Whole-fly simulation is CPU/Brian2 work and can be slow even in its smoke profile.

In [ ]:
# 1. Configuration — edit here before Run all.
PROFILE = 'smoke'                  # 'smoke' or 'confirmation'
CONFIRM_FULL_RUN = False           # True only for the full GPU study
USE_DRIVE = False                  # Colab persistent data/checkpoints
INSTALL_DEPENDENCIES = True
DOWNLOAD_AUDIO = False             # confirmation automatically enables this
DOWNLOAD_REFERENCES = False        # all pinned upstream code + open arXiv PDFs
DOWNLOAD_TIMING_DATA = False       # Zenodo dataset, checksum-verified
RUN_FITS_REFERENCE = False         # 100 epochs per dataset, validation-only selection
RUN_FLY_REFERENCE = False          # whole-brain reference diagnostic, not our next mechanism
FLY_PROFILE = 'smoke'              # 'reference' = author-like 30 × 1-second trials
DOWNLOAD_RESULT_ZIP = False        # automatic browser download in Colab
TIMING_DOWNLOAD_LIMIT_GB = 5.0      # fail before download if exceeded
TIMING_FILE_NAMES = None           # None = all Zenodo files; or exact listed names
LOCAL_WORK_DIR = None              # optional persistent path outside repository

In [ ]:
# 2. Environment and dependencies. Keep an existing GPU PyTorch installation.
import sys, subprocess, importlib.util, os, json, hashlib, base64
from pathlib import Path
def module_available(name):
    try:
        return importlib.util.find_spec(name) is not None
    except (ImportError, ModuleNotFoundError, ValueError):
        return False
IN_COLAB = module_available('google.colab')
def command(args, cwd=None):
    print('Running:', ' '.join(map(str, args)))
    subprocess.run(list(map(str, args)), cwd=cwd, check=True)
if INSTALL_DEPENDENCIES:
    if importlib.util.find_spec('torch') is None:
        command([sys.executable, '-m', 'pip', 'install', 'torch>=2.2,<3'])
    command([sys.executable, '-m', 'pip', 'install', 'numpy>=1.26,<3', 'scipy>=1.11,<2',
             'h5py>=3.10,<4', 'pandas>=2.1,<4', 'matplotlib>=3.8,<4'])
    if RUN_FITS_REFERENCE:
        command([sys.executable, '-m', 'pip', 'install', 'pyyaml>=6,<7'])
    if RUN_FLY_REFERENCE:
        command([sys.executable, '-m', 'pip', 'install', 'brian2>=2.7,<3',
                 'pyarrow>=15,<25', 'joblib>=1.3,<2', 'brotli>=1,<2'])
import torch, numpy as np, pandas as pd
print('Python:', sys.version, '\nPyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unavailable')
assert PROFILE in ('smoke', 'confirmation')
if PROFILE == 'confirmation':
    assert CONFIRM_FULL_RUN, 'Set CONFIRM_FULL_RUN=True to acknowledge the full study.'
    assert torch.cuda.is_available(), 'Select a GPU runtime; CPU runs cannot satisfy the systems gate.'

In [ ]:
# 3. Persistent working directory and a pinned dependency checkout.
BASE_REVISION = 'ee150bd1b0051830f8e3e959805374f4efac9deb'
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = Path('/content/drive/MyDrive/A-SNN-research-restart')
else:
    WORK = Path(LOCAL_WORK_DIR).expanduser().resolve() if LOCAL_WORK_DIR else Path.cwd() / 'research_restart'
WORK.mkdir(parents=True, exist_ok=True)
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
def pinned_checkout(url, revision, path):
    path = Path(path).resolve()
    if not path.exists():
        command(['git', 'clone', '--no-checkout', url, path])
        command(['git', '-C', path, 'checkout', '--detach', revision])
    actual = subprocess.check_output(['git', '-C', str(path), 'rev-parse', 'HEAD'], text=True).strip()
    dirty = subprocess.check_output(['git', '-C', str(path), 'status', '--porcelain', '--untracked-files=no'], text=True).strip()
    if actual != revision or dirty:
        raise RuntimeError(f'Checkout must be clean at {revision}: {path}. Choose a fresh folder; no reset is performed.')
    return path
# Existing local A-SNN can be used without modifying its Git state.
candidates = [Path.cwd(), *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'gen5/ammc_gen5/shd_benchmark.py').is_file()), None)
if REPO is None:
    REPO = pinned_checkout('https://github.com/FaisalTabrez/A-SNN.git', BASE_REVISION,
                           Path('/content/A-SNN-evidence1') if IN_COLAB else WORK / 'A-SNN')
REPO = REPO.resolve()
DATA = WORK / 'data'
OUTPUT = WORK / ('evidence1_v2_' + PROFILE)
print('Repository:', REPO, '\nPersistent work:', WORK, '\nResults:', OUTPUT)

## Portable implementation snapshot
This notebook includes our new runner and adapters, so it works even when
the pinned GitHub base predates them. Each embedded file is SHA-256 checked.
It will create missing files but **refuses to overwrite a different local
implementation**. Use a fresh checkout if you intentionally changed the code.
Original A-SNN modules come from the checkout; all loaded dependency hashes
are recorded in the run protocol. Local checkouts need not equal the pinned base.

In [ ]:
SNAPSHOTS = {'gen5/ammc_gen5/evidence1_audio.py': {'sha256': '94e3dce30bbab3373fe09c78ca2ff365c922ef6fbf3c3788019e5c755ebadcab', 'base64': '...'}, 'gen5/ammc_gen5/artifact_integrity.py': {'sha256': 'a7ebf70f477c1a988734730b41f790a156e43fbf11ebf08780a85cf9bdc88826', 'base64': 'IiIiRmFpbC1jbG9zZWQgbG9jYWwgcHJvdmVuYW5jZSByZWNlaXB0cyAoaW50ZWdyaXR5LCBub3QgZGlnaXRhbCBzaWduYXR1cmVzKS4iIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwppbXBvcnQgaGFzaGxpYgppbXBvcnQgaW1wb3J0bGliLm1ldGFkYXRhCmltcG9ydCBqc29uCmltcG9ydCBwbGF0Zm9ybQppbXBvcnQgc3VicHJvY2Vzcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCgpkZWYgc2hhMjU2X2ZpbGUocGF0aCk6CiAgICB2YWx1ZSA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggUGF0aChwYXRoKS5vcGVuKCdyYicpIGFzIHN0cmVhbToKICAgICAgICBmb3IgYmxvY2sgaW4gaXRlcihsYW1iZGE6IHN0cmVhbS5yZWFkKDEwMjQgKiAxMDI0KSwgYicnKToKICAgICAgICAgICAgdmFsdWUudXBkYXRlKGJsb2NrKQogICAgcmV0dXJuIHZhbHVlLmhleGRpZ2VzdCgpCgoKZGVmIGlkZW50aXR5KHZhbHVlKToKICAgIHJldHVybiBoYXNobGliLnNoYTI1Nihqc29uLmR1bXBzKHZhbHVlLCBzb3J0X2tleXM9VHJ1ZSwgYWxsb3dfbmFuPUZhbHNlKS5lbmNvZGUoKSkuaGV4ZGlnZXN0KCkKCgpkZWYgYXRvbWljX2pzb24ocGF0aCwgdmFsdWUpOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRlbXBvcmFyeSA9IHBhdGgud2l0aF9uYW1lKHBhdGgubmFtZSArICcudG1wJykKICAgIHRlbXBvcmFyeS53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUsIGluZGVudD0yLCBhbGxvd19uYW49RmFsc2UpICsgJ1xuJywgZW5jb2Rpbmc9J3V0Zi04JykKICAgIHRlbXBvcmFyeS5yZXBsYWNlKHBhdGgpCgoKZGVmIGJpbmRfcHJvdG9jb2wocGF0aCwgdmFsdWUsIGxhYmVsPSdQcm90b2NvbCcpOgogICAgIiIiTmV2ZXIgYXR0YWNoIGEgbmV3IHByb3RvY29sIHRvIGEgcHJlLWV4aXN0aW5nIG5vbmVtcHR5IGRpcmVjdG9yeS4iIiIKICAgIHBhdGggPSBQYXRoKHBhdGgpCiAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBpZiBwYXRoLmV4aXN0cygpOgogICAgICAgIGlmIGlkZW50aXR5KGpzb24ubG9hZHMocGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9J3V0Zi04JykpKSAhPSBpZGVudGl0eSh2YWx1ZSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZid7bGFiZWx9IGNoYW5nZWQ7IGNob29zZSBhIG5ldyBvdXRwdXQgZGlyZWN0b3J5JykKICAgIGVsaWYgYW55KHBhdGgucGFyZW50Lml0ZXJkaXIoKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ1VuYm91bmQgbm9uZW1wdHkge2xhYmVsfSBkaXJlY3Rvcnk7IGNob29zZSBhIGZyZXNoIGRpcmVjdG9yeScpCiAgICBlbHNlOgogICAgICAgIGF0b21pY19qc29uKHBhdGgsIHZhbHVlKQoKCmRlZiBfZmlsZXMocm9vdCwgbmFtZXMpOgogICAgcm9vdCA9IFBhdGgocm9vdCkucmVzb2x2ZSgpCiAgICByZXN1bHQgPSB7fQogICAgZm9yIG5hbWUgaW4gbmFtZXM6CiAgICAgICAgb3JpZ2luYWwgPSByb290IC8gbmFtZQogICAgICAgIHBhdGggPSBvcmlnaW5hbC5yZXNvbHZlKCkKICAgICAgICBpZiBvcmlnaW5hbC5pc19zeW1saW5rKCkgb3Igbm90IHBhdGguaXNfcmVsYXRpdmVfdG8ocm9vdCkgb3Igbm90IHBhdGguaXNfZmlsZSgpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYnVW5zYWZlIG9yIG1pc3NpbmcgYXJ0aWZhY3Q6IHtuYW1lfScpCiAgICAgICAgcmVzdWx0W25hbWVdID0geydzaGEyNTYnOiBzaGEyNTZfZmlsZShwYXRoKSwgJ2J5dGVzJzogcGF0aC5zdGF0KCkuc3Rfc2l6ZX0KICAgIHJldHVybiByZXN1bHQKCgpkZWYgc2VhbF9maWxlcyhyb290LCBuYW1lcywgcmVjZWlwdCwgYmluZGluZyk6CiAgICBhdG9taWNfanNvbihQYXRoKHJvb3QpIC8gcmVjZWlwdCwgeydzY2hlbWEnOiAyLCAnYmluZGluZyc6IGJpbmRpbmcsICdmaWxlcyc6IF9maWxlcyhyb290LCBuYW1lcyl9KQoKCmRlZiB2ZXJpZnlfZmlsZXMocm9vdCwgbmFtZXMsIHJlY2VpcHQsIGJpbmRpbmcpOgogICAgcGF0aCA9IFBhdGgocm9vdCkgLyByZWNlaXB0CiAgICBpZiBub3QgcGF0aC5pc19maWxlKCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ01pc3NpbmcgaW50ZWdyaXR5IHJlY2VpcHQ6IHtwYXRofTsgdW5hdXRoZW50aWNhdGVkIGFydGlmYWN0cyBjYW5ub3QgcmVzdW1lJykKICAgIHNhdmVkID0ganNvbi5sb2FkcyhwYXRoLnJlYWRfdGV4dChlbmNvZGluZz0ndXRmLTgnKSkKICAgIGV4cGVjdGVkID0geydzY2hlbWEnOiAyLCAnYmluZGluZyc6IGJpbmRpbmcsICdmaWxlcyc6IF9maWxlcyhyb290LCBuYW1lcyl9CiAgICBpZiBzYXZlZCAhPSBleHBlY3RlZDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYnQXJ0aWZhY3QgaW50ZWdyaXR5L2JpbmRpbmcgbWlzbWF0Y2g6IHtwYXRofTsgcHJlc2VydmUgaXQgYW5kIHVzZSBhIGZyZXNoIGRpcmVjdG9yeScpCgoKZGVmIHJ1bnRpbWVfaWRlbnRpdHkodG9yY2g9Tm9uZSk6CiAgICBwYWNrYWdlcyA9IHtkLm1ldGFkYXRhWydOYW1lJ10ubG93ZXIoKTogZC52ZXJzaW9uIGZvciBkIGluIGltcG9ydGxpYi5tZXRhZGF0YS5kaXN0cmlidXRpb25zKCkKICAgICAgICAgICAgICAgIGlmIGQubWV0YWRhdGEuZ2V0KCdOYW1lJyl9CiAgICByZXN1bHQgPSB7J3B5dGhvbic6IHBsYXRmb3JtLnB5dGhvbl92ZXJzaW9uKCksICdwbGF0Zm9ybSc6IHBsYXRmb3JtLnBsYXRmb3JtKCksCiAgICAgICAgICAgICAgJ3BhY2thZ2VzJzogZGljdChzb3J0ZWQocGFja2FnZXMuaXRlbXMoKSkpfQogICAgaWYgdG9yY2ggaXMgbm90IE5vbmU6CiAgICAgICAgcmVzdWx0LnVwZGF0ZSh0b3JjaD1zdHIodG9yY2guX192ZXJzaW9uX18pLCBjdWRhPXRvcmNoLnZlcnNpb24uY3VkYSwKICAgICAgICAgICAgICAgICAgICAgIGN1ZG5uPXRvcmNoLmJhY2tlbmRzLmN1ZG5uLnZlcnNpb24oKSwKICAgICAgICAgICAgICAgICAgICAgIGRldGVybWluaXN0aWNfYWxnb3JpdGhtcz10b3JjaC5hcmVfZGV0ZXJtaW5pc3RpY19hbGdvcml0aG1zX2VuYWJsZWQoKSwKICAgICAgICAgICAgICAgICAgICAgIGN1ZG5uX2JlbmNobWFyaz10b3JjaC5iYWNrZW5kcy5jdWRubi5iZW5jaG1hcmssCiAgICAgICAgICAgICAgICAgICAgICBjdWRubl9kZXRlcm1pbmlzdGljPXRvcmNoLmJhY2tlbmRzLmN1ZG5uLmRldGVybWluaXN0aWMsCiAgICAgICAgICAgICAgICAgICAgICBtYXRtdWxfdGYzMj10b3JjaC5iYWNrZW5kcy5jdWRhLm1hdG11bC5hbGxvd190ZjMyLAogICAgICAgICAgICAgICAgICAgICAgY3Vkbm5fdGYzMj10b3JjaC5iYWNrZW5kcy5jdWRubi5hbGxvd190ZjMyKQogICAgICAgIGRyaXZlciA9IE5vbmUKICAgICAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBkcml2ZXIgPSBzdWJwcm9jZXNzLmNoZWNrX291dHB1dChbJ252aWRpYS1zbWknLCAnLS1xdWVyeS1ncHU9ZHJpdmVyX3ZlcnNpb24nLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICctLWZvcm1hdD1jc3Ysbm9oZWFkZXInXSwgdGV4dD1UcnVlLCB0aW1lb3V0PTEwKS5zdHJpcCgpCiAgICAgICAgICAgIGV4Y2VwdCAoT1NFcnJvciwgc3VicHJvY2Vzcy5TdWJwcm9jZXNzRXJyb3IpOgogICAgICAgICAgICAgICAgZHJpdmVyID0gJ3VuYXZhaWxhYmxlJwogICAgICAgIHJlc3VsdFsnZHJpdmVyJ10gPSBkcml2ZXIKICAgIHJldHVybiByZXN1bHQK'}, 'gen5/examples/evidence1_audio.py': {'sha256': '4aed9e237b15e5a7554e1a2c3bf72950de00c70711d9fcd9d2528f5a1d04c0ec', 'base64': 'IiIiUnVuIHRoZSBmcm96ZW4gRXZpZGVuY2UtMSBhdWRpbyBjb25maXJtYXRpb24gb3IgYSBzeW50aGV0aWMgc21va2UgY2hlY2suIiIiCmltcG9ydCBzeXMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzFdKSkKZnJvbSBhbW1jX2dlbjUuZXZpZGVuY2UxX2F1ZGlvIGltcG9ydCBtYWluCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=='}, 'gen5/tools/fits_validation_reference.py': {'sha256': '2ede240f48ac1f34202ea6d04fd1fe6a13da332403b3bdd20a02e509ceb9ad9d', 'base64': 'IiIiUGlubmVkIEZpVFMgbmF0aXZlLXJlY2lwZSByZWZlcmVuY2UsIHNlbGVjdGluZyBvbiB2YWxpZGF0aW9uIGFuZCB0ZXN0aW5nIG9uY2UuCgpUaGlzIGlzIG5vdCBhbiBFdmlkZW5jZS0xIG1hdGNoZWQgY29udHJvbC4gUnVuIGluIGEgc3VicHJvY2VzczsgdGhlIGFkYXB0ZXIKcmVwbGFjZXMgb25seSB0aGUgdXBzdHJlYW0gZml0IGZ1bmN0aW9uIGFuZCBsZWF2ZXMgaXRzIGRhdGEvbW9kZWwgdW5jaGFuZ2VkLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBoYXNobGliCmltcG9ydCBpbXBvcnRsaWIKaW1wb3J0IGpzb24KaW1wb3J0IHJhbmRvbQppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzFdKSkKZnJvbSBhbW1jX2dlbjUuYXJ0aWZhY3RfaW50ZWdyaXR5IGltcG9ydCBiaW5kX3Byb3RvY29sLCBpZGVudGl0eSwgcnVudGltZV9pZGVudGl0eSwgc2VhbF9maWxlcywgdmVyaWZ5X2ZpbGVzCgpGSVRTX1JFVklTSU9OID0gIjhmZTdlMzc2YmFlMTNmZTc1ZjM1N2RjZmJmZjZhZGY5YzQ1ZjEwNzciCgoKZGVmIHNoYTI1NihwYXRoKToKICAgIGggPSBoYXNobGliLnNoYTI1NigpCiAgICB3aXRoIFBhdGgocGF0aCkub3BlbigicmIiKSBhcyBzdHJlYW06CiAgICAgICAgZm9yIGNodW5rIGluIGl0ZXIobGFtYmRhOiBzdHJlYW0ucmVhZCgxMDI0ICogMTAyNCksIGIiIik6CiAgICAgICAgICAgIGgudXBkYXRlKGNodW5rKQogICAgcmV0dXJuIGguaGV4ZGlnZXN0KCkKCgpkZWYgd3JpdGVfanNvbihwYXRoLCB2YWx1ZSk6CiAgICBwYXRoID0gUGF0aChwYXRoKQogICAgdGVtcG9yYXJ5ID0gcGF0aC53aXRoX3N1ZmZpeCgiLnRtcCIpCiAgICB0ZW1wb3Jhcnkud3JpdGVfdGV4dChqc29uLmR1bXBzKHZhbHVlLCBpbmRlbnQ9MiwgYWxsb3dfbmFuPUZhbHNlKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgIHRlbXBvcmFyeS5yZXBsYWNlKHBhdGgpCgoKZGVmIHZhbGlkYXRpb25fZml0KG1vZGVsLCBjZmcsIG91dF9kaXIsIGRldmljZSwgdHJhaW5fbG9hZGVyLCB0ZXN0X2xvYWRlciwKICAgICAgICAgICAgICAgICAgIHZhbGlkX2xvYWRlcj1Ob25lLCBwcmVwPU5vbmUsIGV4dHJhX2NvbmZpZz1Ob25lKToKICAgIGlmIHZhbGlkX2xvYWRlciBpcyBOb25lOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkZpbmFsLXRlc3Qgc2VsZWN0aW9uIGlzIGZvcmJpZGRlbjsgdXNlIFNIRCB2YWxpZF9zcGxpdCBvciBTU0Mgb2ZmaWNpYWwgdmFsaWRhdGlvbiIpCiAgICBmcm9tIGNvbW1vbi50cmFpbmluZyBpbXBvcnQgY3JlYXRlX29wdGltaXplciwgcnVuX2Vwb2NoCiAgICBvdXRfZGlyID0gUGF0aChvdXRfZGlyKQogICAgb3V0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBjb25maWd1cmF0aW9uID0geyoqdmFycyhjZmcpLCAqKihleHRyYV9jb25maWcgb3Ige30pfQogICAgY29uZmlnX3BhdGggPSBvdXRfZGlyIC8gInZhbGlkYXRpb25fcmVmZXJlbmNlX2NvbmZpZy5qc29uIgogICAgbWFuYWdlZCA9IFsndmFsaWRhdGlvbl9yZWZlcmVuY2Vfc3VtbWFyeS5qc29uJywgJ3ZhbGlkYXRpb25fcmVmZXJlbmNlX2NoZWNrcG9pbnQucHQnLCAndmFsaWRhdGlvbl9yZWZlcmVuY2VfaGlzdG9yeS5qc29uJ10KICAgIGlmIG5vdCBjb25maWdfcGF0aC5leGlzdHMoKSBhbmQgYW55KChvdXRfZGlyIC8gbmFtZSkuZXhpc3RzKCkgZm9yIG5hbWUgaW4gbWFuYWdlZCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVW5ib3VuZCBGaVRTIGFydGlmYWN0czsgY2hvb3NlIGEgZnJlc2ggb3V0cHV0IGRpcmVjdG9yeScpCiAgICBpZiBjb25maWdfcGF0aC5leGlzdHMoKSBhbmQgaWRlbnRpdHkoanNvbi5sb2Fkcyhjb25maWdfcGF0aC5yZWFkX3RleHQoKSkpICE9IGlkZW50aXR5KGNvbmZpZ3VyYXRpb24pOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkZpVFMgY29uZmlndXJhdGlvbiBjaGFuZ2VkOyB1c2UgYSBuZXcgcmVzdWx0cyBkaXJlY3RvcnkiKQogICAgd3JpdGVfanNvbihjb25maWdfcGF0aCwgY29uZmlndXJhdGlvbikKICAgIGJpbmRpbmcgPSB7J3NjaGVtYSc6IDIsICdjb25maWd1cmF0aW9uJzogaWRlbnRpdHkoY29uZmlndXJhdGlvbiksCiAgICAgICAgICAgICAgICdwcm90b2NvbCc6IHNoYTI1NihvdXRfZGlyIC8gJ3ZhbGlkYXRpb25fcmVmZXJlbmNlX3Byb3RvY29sLmpzb24nKSBpZiAob3V0X2RpciAvICd2YWxpZGF0aW9uX3JlZmVyZW5jZV9wcm90b2NvbC5qc29uJykuZXhpc3RzKCkgZWxzZSBOb25lfQogICAgc3VtbWFyeV9wYXRoID0gb3V0X2RpciAvICJ2YWxpZGF0aW9uX3JlZmVyZW5jZV9zdW1tYXJ5Lmpzb24iCiAgICBpZiBzdW1tYXJ5X3BhdGguZXhpc3RzKCk6CiAgICAgICAgdmVyaWZ5X2ZpbGVzKG91dF9kaXIsIG1hbmFnZWQsICd2YWxpZGF0aW9uX3JlZmVyZW5jZV9jb21wbGV0ZV9pbnRlZ3JpdHkuanNvbicsIGJpbmRpbmcpCiAgICAgICAgc3VtbWFyeSA9IGpzb24ubG9hZHMoc3VtbWFyeV9wYXRoLnJlYWRfdGV4dCgpKQogICAgICAgIGlmIHN1bW1hcnkuZ2V0KCdzZWxlY3Rpb24nKSAhPSAndmFsaWRhdGlvbl9vbmx5JyBvciBzdW1tYXJ5LmdldCgndGVzdF9ldmFsdWF0aW9ucycpICE9IDE6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0ludmFsaWQgRmlUUyBzdW1tYXJ5IGNvbnRyYWN0JykKICAgICAgICByZXR1cm4gc3VtbWFyeQogICAgb3B0aW1pemVyLCBzY2hlZHVsZXIsIGd1YXJkID0gY3JlYXRlX29wdGltaXplcihtb2RlbCwgY2ZnKQogICAgY3JpdGVyaW9uID0gdG9yY2gubm4uQ3Jvc3NFbnRyb3B5TG9zcygpCiAgICBjaGVja3BvaW50ID0gb3V0X2RpciAvICJ2YWxpZGF0aW9uX3JlZmVyZW5jZV9jaGVja3BvaW50LnB0IgogICAgZmlyc3QsIGJlc3QsIGJlc3RfZXBvY2gsIGJlc3Rfc3RhdGUsIGhpc3RvcnkgPSAxLCAtMS4wLCAwLCBOb25lLCBbXQogICAgaWYgY2hlY2twb2ludC5leGlzdHMoKToKICAgICAgICB2ZXJpZnlfZmlsZXMob3V0X2RpciwgW2NoZWNrcG9pbnQubmFtZV0sICd2YWxpZGF0aW9uX3JlZmVyZW5jZV9jaGVja3BvaW50X2ludGVncml0eS5qc29uJywgYmluZGluZykKICAgICAgICBzYXZlZCA9IHRvcmNoLmxvYWQoY2hlY2twb2ludCwgbWFwX2xvY2F0aW9uPSJjcHUiLCB3ZWlnaHRzX29ubHk9RmFsc2UpCiAgICAgICAgaWYgc2F2ZWQuZ2V0KCdiaW5kaW5nJykgIT0gYmluZGluZzoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignRmlUUyBjaGVja3BvaW50IGJpbmRpbmcgY2hhbmdlZCcpCiAgICAgICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KHNhdmVkWyJtb2RlbCJdKQogICAgICAgIG9wdGltaXplci5sb2FkX3N0YXRlX2RpY3Qoc2F2ZWRbIm9wdGltaXplciJdKQogICAgICAgIHNjaGVkdWxlci5sb2FkX3N0YXRlX2RpY3Qoc2F2ZWRbInNjaGVkdWxlciJdKQogICAgICAgIGZpcnN0ID0gc2F2ZWRbImVwb2NoIl0gKyAxCiAgICAgICAgYmVzdCwgYmVzdF9lcG9jaCwgYmVzdF9zdGF0ZSwgaGlzdG9yeSA9IHNhdmVkWyJiZXN0Il0sIHNhdmVkWyJiZXN0X2Vwb2NoIl0sIHNhdmVkWyJiZXN0X3N0YXRlIl0sIHNhdmVkWyJoaXN0b3J5Il0KICAgICAgICByYW5kb20uc2V0c3RhdGUoc2F2ZWRbInJhbmRvbSJdKQogICAgICAgIG5wLnJhbmRvbS5zZXRfc3RhdGUoc2F2ZWRbIm51bXB5Il0pCiAgICAgICAgdG9yY2guc2V0X3JuZ19zdGF0ZShzYXZlZFsidG9yY2giXSkKICAgICAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICB0b3JjaC5jdWRhLnNldF9ybmdfc3RhdGVfYWxsKHNhdmVkWyJjdWRhIl0pCiAgICBmb3IgZXBvY2ggaW4gcmFuZ2UoZmlyc3QsIGNmZy5lcG9jaHMgKyAxKToKICAgICAgICBsb3NzLCBhY2N1cmFjeSA9IHJ1bl9lcG9jaChtb2RlbCwgdHJhaW5fbG9hZGVyLCBjcml0ZXJpb24sIGRldmljZSwgb3B0aW1pemVyLCBwcmVwKQogICAgICAgIF8sIHZhbGlkYXRpb24gPSBydW5fZXBvY2gobW9kZWwsIHZhbGlkX2xvYWRlciwgY3JpdGVyaW9uLCBkZXZpY2UsIHByZXA9cHJlcCkKICAgICAgICBpZiBub3QgbnAuaXNmaW5pdGUobG9zcykgb3Igbm90IG5wLmlzZmluaXRlKHZhbGlkYXRpb24pOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk5vbmZpbml0ZSBGaVRTIHRyYWluaW5nL3ZhbGlkYXRpb24gbWV0cmljIikKICAgICAgICBoaXN0b3J5LmFwcGVuZCh7ImVwb2NoIjogZXBvY2gsICJ0cmFpbl9sb3NzIjogbG9zcywgInRyYWluX2FjYyI6IGFjY3VyYWN5LCAidmFsaWRfYWNjIjogdmFsaWRhdGlvbn0pCiAgICAgICAgaWYgdmFsaWRhdGlvbiA+IGJlc3Q6CiAgICAgICAgICAgIGJlc3QsIGJlc3RfZXBvY2ggPSB2YWxpZGF0aW9uLCBlcG9jaAogICAgICAgICAgICBiZXN0X3N0YXRlID0ge2s6IHYuZGV0YWNoKCkuY3B1KCkuY2xvbmUoKSBmb3IgaywgdiBpbiBtb2RlbC5zdGF0ZV9kaWN0KCkuaXRlbXMoKX0KICAgICAgICBzY2hlZHVsZXIuc3RlcCgpCiAgICAgICAgaWYgZ3VhcmQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGd1YXJkLnJlc2V0X2NvdW50ZXIoKQogICAgICAgIHRlbXBvcmFyeSA9IGNoZWNrcG9pbnQud2l0aF9zdWZmaXgoIi50bXAiKQogICAgICAgIHRvcmNoLnNhdmUoeyJiaW5kaW5nIjogYmluZGluZywgIm1vZGVsIjogbW9kZWwuc3RhdGVfZGljdCgpLCAib3B0aW1pemVyIjogb3B0aW1pemVyLnN0YXRlX2RpY3QoKSwgInNjaGVkdWxlciI6IHNjaGVkdWxlci5zdGF0ZV9kaWN0KCksCiAgICAgICAgICAgICAgICAgICAgImVwb2NoIjogZXBvY2gsICJiZXN0IjogYmVzdCwgImJlc3RfZXBvY2giOiBiZXN0X2Vwb2NoLCAiYmVzdF9zdGF0ZSI6IGJlc3Rfc3RhdGUsICJoaXN0b3J5IjogaGlzdG9yeSwKICAgICAgICAgICAgICAgICAgICAicmFuZG9tIjogcmFuZG9tLmdldHN0YXRlKCksICJudW1weSI6IG5wLnJhbmRvbS5nZXRfc3RhdGUoKSwgInRvcmNoIjogdG9yY2guZ2V0X3JuZ19zdGF0ZSgpLAogICAgICAgICAgICAgICAgICAgICJjdWRhIjogdG9yY2guY3VkYS5nZXRfcm5nX3N0YXRlX2FsbCgpIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSBbXX0sIHRlbXBvcmFyeSkKICAgICAgICB0ZW1wb3JhcnkucmVwbGFjZShjaGVja3BvaW50KQogICAgICAgIHNlYWxfZmlsZXMob3V0X2RpciwgW2NoZWNrcG9pbnQubmFtZV0sICd2YWxpZGF0aW9uX3JlZmVyZW5jZV9jaGVja3BvaW50X2ludGVncml0eS5qc29uJywgYmluZGluZykKICAgICAgICBwcmludChmIkZpVFMgZXBvY2gge2Vwb2NofS97Y2ZnLmVwb2Noc30gdmFsaWRhdGlvbj17dmFsaWRhdGlvbjouNGZ9IiwgZmx1c2g9VHJ1ZSkKICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChiZXN0X3N0YXRlKQogICAgXywgdGVzdCA9IHJ1bl9lcG9jaChtb2RlbCwgdGVzdF9sb2FkZXIsIGNyaXRlcmlvbiwgZGV2aWNlLCBwcmVwPXByZXApCiAgICBpZiBub3QgbnAuaXNmaW5pdGUodGVzdCk6CiAgICAgICAgcmFpc2UgRmxvYXRpbmdQb2ludEVycm9yKCdOb25maW5pdGUgRmlUUyBmaW5hbCBldmFsdWF0aW9uJykKICAgIHN1bW1hcnkgPSB7InNlbGVjdGlvbiI6ICJ2YWxpZGF0aW9uX29ubHkiLCAiYmVzdF9lcG9jaCI6IGJlc3RfZXBvY2gsICJiZXN0X3ZhbGlkYXRpb25fYWNjdXJhY3kiOiBiZXN0LAogICAgICAgICAgICAgICAidGVzdF9hY2N1cmFjeSI6IHRlc3QsICJ0ZXN0X2V2YWx1YXRpb25zIjogMSwgImtpbmQiOiAibmF0aXZlX3JlY2lwZV9ub3RfbWF0Y2hlZF9FdmlkZW5jZTFfY29udHJvbCJ9CiAgICB3cml0ZV9qc29uKG91dF9kaXIgLyAidmFsaWRhdGlvbl9yZWZlcmVuY2VfaGlzdG9yeS5qc29uIiwgaGlzdG9yeSkKICAgIHdyaXRlX2pzb24oc3VtbWFyeV9wYXRoLCBzdW1tYXJ5KQogICAgc2VhbF9maWxlcyhvdXRfZGlyLCBtYW5hZ2VkLCAndmFsaWRhdGlvbl9yZWZlcmVuY2VfY29tcGxldGVfaW50ZWdyaXR5Lmpzb24nLCBiaW5kaW5nKQogICAgcmV0dXJuIHN1bW1hcnkKCgpkZWYgbWFpbigpOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tdXBzdHJlYW0iLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRhdGFzZXQiLCBjaG9pY2VzPSgic2hkIiwgInNzYyIpLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kYXRhLXJvb3QiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dC1kaXIiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNlZWQiLCB0eXBlPWludCwgZGVmYXVsdD0zMTApCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWVwb2NocyIsIHR5cGU9aW50LCBkZWZhdWx0PTEwMCkKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICB1cHN0cmVhbSA9IGFyZ3MudXBzdHJlYW0ucmVzb2x2ZSgpCiAgICBhY3R1YWwgPSBzdWJwcm9jZXNzLmNoZWNrX291dHB1dChbImdpdCIsICItQyIsIHN0cih1cHN0cmVhbSksICJyZXYtcGFyc2UiLCAiSEVBRCJdLCB0ZXh0PVRydWUpLnN0cmlwKCkKICAgIGRpcnR5ID0gc3VicHJvY2Vzcy5jaGVja19vdXRwdXQoWyJnaXQiLCAiLUMiLCBzdHIodXBzdHJlYW0pLCAic3RhdHVzIiwgIi0tcG9yY2VsYWluIiwgIi0tdW50cmFja2VkLWZpbGVzPW5vIl0sIHRleHQ9VHJ1ZSkuc3RyaXAoKQogICAgaWYgYWN0dWFsICE9IEZJVFNfUkVWSVNJT04gb3IgZGlydHk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRmlUUyBjaGVja291dCBtdXN0IGJlIGNsZWFuIGF0IHRoZSBwaW5uZWQgcmV2aXNpb24iKQogICAgYXJncy5vdXRwdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGRhdGFzZXQgPSBhcmdzLmRhdGFzZXQudXBwZXIoKQogICAgcmVjaXBlID0gdXBzdHJlYW0gLyAiY29uZmlnIiAvIGRhdGFzZXQgLyAoImZpdHNfbzJfdzI1Ni55YW1sIiBpZiBkYXRhc2V0ID09ICJTSEQiIGVsc2UgImZpdHNfbzFfdzUxMi55YW1sIikKICAgIGRhdGEgPSBhcmdzLmRhdGFfcm9vdC5yZXNvbHZlKCkgLyBhcmdzLmRhdGFzZXQKICAgIGZpbGVzID0ge25hbWU6IHNoYTI1NihkYXRhIC8gbmFtZSkgZm9yIG5hbWUgaW4KICAgICAgICAgICAgIChbInNoZF90cmFpbi5oNSIsICJzaGRfdGVzdC5oNSJdIGlmIGRhdGFzZXQgPT0gIlNIRCIgZWxzZSBbInNzY190cmFpbi5oNSIsICJzc2NfdmFsaWQuaDUiLCAic3NjX3Rlc3QuaDUiXSl9CiAgICBzaWduYXR1cmUgPSB7InNjaGVtYSI6IDIsICJydW50aW1lIjogcnVudGltZV9pZGVudGl0eSh0b3JjaCksICJ1cHN0cmVhbSI6IGFjdHVhbCwgImFkYXB0ZXIiOiBzaGEyNTYoX19maWxlX18pLCAicmVjaXBlIjogc2hhMjU2KHJlY2lwZSksICJyYXdfZGF0YV9zaGEyNTYiOiBmaWxlcywKICAgICAgICAgICAgICAgICAidG9yY2giOiBzdHIodG9yY2guX192ZXJzaW9uX18pLCAicHl0aG9uIjogc3lzLnZlcnNpb24sICJzZWVkIjogYXJncy5zZWVkLCAiZXBvY2hzIjogYXJncy5lcG9jaHMsCiAgICAgICAgICAgICAgICAgImRldmljZSI6IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1In0KICAgIGxvY2sgPSBhcmdzLm91dHB1dF9kaXIgLyAidmFsaWRhdGlvbl9yZWZlcmVuY2VfcHJvdG9jb2wuanNvbiIKICAgIGJpbmRfcHJvdG9jb2wobG9jaywgc2lnbmF0dXJlLCAnRmlUUyBwcm90b2NvbC9lbnZpcm9ubWVudC9kYXRhJykKICAgIHN5cy5wYXRoLmluc2VydCgwLCBzdHIodXBzdHJlYW0pKQogICAgbW9kdWxlID0gaW1wb3J0bGliLmltcG9ydF9tb2R1bGUoZiJ7ZGF0YXNldH0uc2NyaXB0cy50cmFpbi50cmFpbl9jaHVua2VkIikKICAgIG1vZHVsZS5maXQgPSB2YWxpZGF0aW9uX2ZpdAogICAgc3lzLmFyZ3YgPSBbImZpdHNfcmVmZXJlbmNlIiwgIi0tY29uZmlnIiwgc3RyKHJlY2lwZSksICItLWRhdGEtZGlyIiwgc3RyKGRhdGEpLAogICAgICAgICAgICAgICAgIi0tcmVzdWx0cy1kaXIiLCBzdHIoYXJncy5vdXRwdXRfZGlyLnJlc29sdmUoKSksICItLXNlZWQiLCBzdHIoYXJncy5zZWVkKSwKICAgICAgICAgICAgICAgICItLWVwb2NocyIsIHN0cihhcmdzLmVwb2NocyksICItLXdvcmtlcnMiLCAiMCIsICItLWJhY2tlbmQiLCAidG9yY2giXQogICAgaWYgZGF0YXNldCA9PSAiU0hEIjoKICAgICAgICBzeXMuYXJndi5leHRlbmQoWyItLWV2YWwtbW9kZSIsICJ2YWxpZF9zcGxpdCJdKQogICAgbW9kdWxlLm1haW4oKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK'}, 'gen5/tools/fly_reference.py': {'sha256': 'ffb8c3c764924f4a096ae14c604c969e7db3b7dbe5c9d2a248e756bd0b33cd94', 'base64': 'IiIiUnVuIGEgcGlubmVkIHdob2xlLWZseSByZWZlcmVuY2UgZGlhZ25vc3RpYywgbm90IGFuIEEtU05OIG1lY2hhbmlzbSBjbGFpbS4iIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXSkpCmZyb20gYW1tY19nZW41LmFydGlmYWN0X2ludGVncml0eSBpbXBvcnQgYmluZF9wcm90b2NvbCwgaWRlbnRpdHksIHJ1bnRpbWVfaWRlbnRpdHksIHNlYWxfZmlsZXMsIHZlcmlmeV9maWxlcwoKUkVWSVNJT04gPSAiOTFiZGQxZTdkY2YxOTNmM2U3Y2E1YTg5MzM0OTdmY2VmNjNiNzk2MCIKU1VHQVIgPSBbNzIwNTc1OTQwNjI0OTYzNzg2LCA3MjA1NzU5NDA2MzAyMzM5MTYsIDcyMDU3NTk0MDYzNzU2ODgzOCwKICAgICAgICAgNzIwNTc1OTQwNjM4MjAyMzQ1LCA3MjA1NzU5NDA2MTcwMDA3NjgsIDcyMDU3NTk0MDYzMDc5NzExMywKICAgICAgICAgNzIwNTc1OTQwNjMyODg5Mzg5LCA3MjA1NzU5NDA2MjE3NTQzNjcsIDcyMDU3NTk0MDYyMTUwMjA1MSwKICAgICAgICAgNzIwNTc1OTQwNjQwNjQ5NjkxLCA3MjA1NzU5NDA2MzkzMzI3MzYsIDcyMDU3NTk0MDYxNjg4NTUzOCwKICAgICAgICAgNzIwNTc1OTQwNjM5MTk4NjUzLCA3MjA1NzU5NDA2MjA5MDA0NDYsIDcyMDU3NTk0MDYxNzkzNzU0MywKICAgICAgICAgNzIwNTc1OTQwNjMyNDI1OTE5LCA3MjA1NzU5NDA2MzMxNDM4MzMsIDcyMDU3NTk0MDYxMjY3MDU3MCwKICAgICAgICAgNzIwNTc1OTQwNjI4ODUzMjM5LCA3MjA1NzU5NDA2MjkxNzY2NjMsIDcyMDU3NTk0MDYxMTg3NTU3MF0KCgpkZWYgbWFpbigpOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tdXBzdHJlYW0iLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dC1kaXIiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXByb2ZpbGUiLCBjaG9pY2VzPSgic21va2UiLCAicmVmZXJlbmNlIiksIGRlZmF1bHQ9InNtb2tlIikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICByb290LCBvdXRwdXQgPSBhcmdzLnVwc3RyZWFtLnJlc29sdmUoKSwgYXJncy5vdXRwdXRfZGlyLnJlc29sdmUoKQogICAgaGVhZCA9IHN1YnByb2Nlc3MuY2hlY2tfb3V0cHV0KFsiZ2l0IiwgIi1DIiwgc3RyKHJvb3QpLCAicmV2LXBhcnNlIiwgIkhFQUQiXSwgdGV4dD1UcnVlKS5zdHJpcCgpCiAgICBkaXJ0eSA9IHN1YnByb2Nlc3MuY2hlY2tfb3V0cHV0KFsiZ2l0IiwgIi1DIiwgc3RyKHJvb3QpLCAic3RhdHVzIiwgIi0tcG9yY2VsYWluIiwgIi0tdW50cmFja2VkLWZpbGVzPW5vIl0sIHRleHQ9VHJ1ZSkuc3RyaXAoKQogICAgaWYgaGVhZCAhPSBSRVZJU0lPTiBvciBkaXJ0eToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJGbHkgc291cmNlL2RhdGEgbXVzdCBiZSBjbGVhbiBhdCB0aGUgcGlubmVkIHJldmlzaW9uIikKICAgIGltcG9ydCBicmlhbjIgYXMgYjIKICAgIGltcG9ydCBwYW5kYXMgYXMgcGQKICAgIHN5cy5wYXRoLmluc2VydCgwLCBzdHIocm9vdCkpCiAgICBpbXBvcnQgbW9kZWwKICAgIGltcG9ydCB1dGlscwogICAgZmlsZXMgPSBbcm9vdCAvICIyMDIzXzAzXzIzX2NvbXBsZXRlbmVzc182MzBfZmluYWwuY3N2Iiwgcm9vdCAvICIyMDIzXzAzXzIzX2Nvbm5lY3Rpdml0eV82MzBfZmluYWwucGFycXVldCJdCiAgICBvdXRwdXQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgIyBQeXRob24gY29kZWdlbiBpcyBwb3J0YWJsZSBhbmQgYXZvaWRzIHNpbGVudGx5IGluc3RhbGxpbmcgYSBDKysgdG9vbGNoYWluLgogICAgIyBJdCBpcyBzbG93ZXIgdGhhbiB0aGUgYXV0aG9yJ3MgY29tcGlsZWQgZW52aXJvbm1lbnQ7IG5vIHRocm91Z2hwdXQgY2xhaW0uCiAgICBiMi5wcmVmcy5jb2RlZ2VuLnRhcmdldCA9ICJudW1weSIKICAgIG4sIGR1cmF0aW9uID0gKDEsIDAuMSkgaWYgYXJncy5wcm9maWxlID09ICJzbW9rZSIgZWxzZSAoMzAsIDEuMCkKICAgIGxvY2sgPSB7InNjaGVtYSI6IDIsICJydW50aW1lIjogcnVudGltZV9pZGVudGl0eSgpLCAia2luZCI6ICJyZWZlcmVuY2VfZGlhZ25vc3RpY19ub3RfQVNOTl9tZWNoYW5pc20iLCAidXBzdHJlYW0iOiBoZWFkLAogICAgICAgICAgICAicHJvZmlsZSI6IGFyZ3MucHJvZmlsZSwgInNlZWQiOiAzMTAsICJuX3J1biI6IG4sICJkdXJhdGlvbl9zZWNvbmRzIjogZHVyYXRpb24sCiAgICAgICAgICAgICJicmlhbjIiOiBiMi5fX3ZlcnNpb25fXywgInB5dGhvbiI6IHN5cy52ZXJzaW9uLCAiY29kZWdlbiI6ICJudW1weSIsCiAgICAgICAgICAgICJhZGFwdGVyX3NoYTI1NiI6IGhhc2hsaWIuc2hhMjU2KFBhdGgoX19maWxlX18pLnJlYWRfYnl0ZXMoKSkuaGV4ZGlnZXN0KCksCiAgICAgICAgICAgICJpbnB1dHNfc2hhMjU2Ijoge3AubmFtZTogaGFzaGxpYi5zaGEyNTYocC5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpIGZvciBwIGluIGZpbGVzfX0KICAgIHByb3RvY29sID0gb3V0cHV0IC8gImZseV9wcm90b2NvbC5qc29uIgogICAgYmluZF9wcm90b2NvbChwcm90b2NvbCwgbG9jaywgJ0ZseSBwcm90b2NvbCcpCiAgICBmb3IgbmFtZSwgaHogaW4gWygic3VnYXJSIiwgMTUwKSwgKCJzdWdhclJfMTAwSHoiLCAxMDApXToKICAgICAgICBiaW5kaW5nID0geydwcm90b2NvbCc6IGlkZW50aXR5KGxvY2spLCAnZXhwZXJpbWVudCc6IG5hbWUsICdyYXRlX2h6JzogaHp9CiAgICAgICAgaWYgKG91dHB1dCAvIChuYW1lICsgJy5wYXJxdWV0JykpLmV4aXN0cygpOgogICAgICAgICAgICB2ZXJpZnlfZmlsZXMob3V0cHV0LCBbbmFtZSArICcucGFycXVldCddLCBuYW1lICsgJ19pbnRlZ3JpdHkuanNvbicsIGJpbmRpbmcpCiAgICAgICAgYjIuc2VlZCgzMTApCiAgICAgICAgcGFyYW1zID0gZGljdChtb2RlbC5kZWZhdWx0X3BhcmFtcywgbl9ydW49biwgdF9ydW49ZHVyYXRpb24gKiBiMi5zZWNvbmQsIHJfcG9pPWh6ICogYjIuSHopCiAgICAgICAgbW9kZWwucnVuX2V4cChuYW1lLCBTVUdBUiwgcGF0aF9yZXM9c3RyKG91dHB1dCksIHBhdGhfY29tcD1zdHIoZmlsZXNbMF0pLCBwYXRoX2Nvbj1zdHIoZmlsZXNbMV0pLAogICAgICAgICAgICAgICAgICAgICAgcGFyYW1zPXBhcmFtcywgbl9wcm9jPTEsIGZvcmNlX292ZXJ3cml0ZT1GYWxzZSkKICAgICAgICBzZWFsX2ZpbGVzKG91dHB1dCwgW25hbWUgKyAnLnBhcnF1ZXQnXSwgbmFtZSArICdfaW50ZWdyaXR5Lmpzb24nLCBiaW5kaW5nKQogICAgY29tYmluZWQgPSBwZC5jb25jYXQoW3BkLnJlYWRfcGFycXVldChvdXRwdXQgLyAobmFtZSArICIucGFycXVldCIpKSBmb3IgbmFtZSBpbiAoInN1Z2FyUiIsICJzdWdhclJfMTAwSHoiKV0pCiAgICByYXRlcywgZGV2aWF0aW9ucyA9IHV0aWxzLmdldF9yYXRlKGNvbWJpbmVkLCBkdXJhdGlvbiwgbikKICAgIHJhdGVzLnRvX2NzdihvdXRwdXQgLyAiZmlyaW5nX3JhdGVzX2h6LmNzdiIpCiAgICBkZXZpYXRpb25zLnRvX2NzdihvdXRwdXQgLyAiZmlyaW5nX3JhdGVzX3N0ZF9oei5jc3YiKQogICAgIyBDb21wYXJlIGF1dGhvciBleGFtcGxlcyBkZXNjcmlwdGl2ZWx5IG9ubHk6IGRpZmZlcmVudCBzdG9jaGFzdGljIHRyaWFscy4KICAgIGF1dGhvciA9IHBkLmNvbmNhdChbcGQucmVhZF9wYXJxdWV0KHJvb3QgLyAicmVzdWx0cyIgLyAiZXhhbXBsZSIgLyAobmFtZSArICIucGFycXVldCIpKQogICAgICAgICAgICAgICAgICAgICAgICBmb3IgbmFtZSBpbiAoInN1Z2FyUiIsICJzdWdhclJfMTAwSHoiKV0pCiAgICBhdXRob3JfcmF0ZXMsIF8gPSB1dGlscy5nZXRfcmF0ZShhdXRob3IsIDEuMCwgMzApCiAgICBhdXRob3JfcmF0ZXMudG9fY3N2KG91dHB1dCAvICJhdXRob3JfZXhhbXBsZV9yYXRlc19oei5jc3YiKQogICAgdGFyZ2V0ID0gNzIwNTc1OTQwNjYwMjE5MjY1CiAgICBwcmludCgiTU45IHJlZmVyZW5jZSByZWFkb3V0IChIejsgYWJzZW50IHNwaWtpbmcgcm93cyBhcmUgemVybyk6IikKICAgIHByaW50KHJhdGVzLnJlaW5kZXgoW3RhcmdldF0pLmZpbGxuYSgwKSkKICAgIHByaW50KCJBdXRob3IgZXhhbXBsZSAoZGlmZmVyZW50IHN0b2NoYXN0aWMgcnVucyk6IikKICAgIHByaW50KGF1dGhvcl9yYXRlcy5yZWluZGV4KFt0YXJnZXRdKS5maWxsbmEoMCkpCiAgICBwcmludCgiTm8gcGFwZXItcmVwcm9kdWN0aW9uIG9yIEEtU05OIGNhdXNhbCBnYXRlIGlzIGNsYWltZWQgYnkgdGhpcyBkaWFnbm9zdGljLiIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='}}
for relative, snapshot in SNAPSHOTS.items():
    content = base64.b64decode(snapshot['base64'])
    assert hashlib.sha256(content).hexdigest() == snapshot['sha256']
    path = REPO / relative
    if path.exists():
        assert sha256(path) == snapshot['sha256'], f'Different existing file: {path}. Use a fresh checkout.'
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(content)
sys.path.insert(0, str(REPO / 'gen5'))
from ammc_gen5.evidence1_audio import EvidenceConfig, make_model, ARMS
from ammc_gen5.artifact_integrity import atomic_json, runtime_identity
atomic_json(WORK / 'runtime_freeze.json', runtime_identity(torch))
config = EvidenceConfig.confirmation() if PROFILE == 'confirmation' else EvidenceConfig()
print(config)
for dataset, classes in [('shd', 20), ('ssc', 35)]:
    if PROFILE == 'smoke':
        classes, inputs, duration = 3, 12, 1.0
    else:
        inputs, duration = 700, 1.4 if dataset == 'shd' else 1.0
    print(dataset, {arm: sum(p.numel() for p in make_model(arm, config, inputs, classes, duration).parameters()) for arm in ARMS})

In [ ]:
# 4. Official SHD/SSC download — MD5 archive check plus raw SHA-256 provenance.
# No downloads are needed for synthetic smoke. Raw data is NOT in the result ZIP.
import urllib.request, gzip, shutil
def fetch(url, destination, checksum=None, algorithm='sha256'):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    def digest(path):
        h = hashlib.new(algorithm)
        with path.open('rb') as f:
            for block in iter(lambda: f.read(1024 * 1024), b''): h.update(block)
        return h.hexdigest()
    if destination.exists():
        if checksum and digest(destination) != checksum:
            raise ValueError(f'Checksum mismatch for existing {destination}; move it aside and retry.')
        return destination
    temporary = destination.with_suffix(destination.suffix + '.part')
    print('Downloading', url)
    request = urllib.request.Request(url, headers={'User-Agent': 'A-SNN-research-notebook/1.0'})
    with urllib.request.urlopen(request, timeout=120) as response, temporary.open('wb') as target:
        shutil.copyfileobj(response, target)
    if checksum and digest(temporary) != checksum:
        raise ValueError(f'Checksum mismatch: {temporary}; destination was not replaced.')
    temporary.replace(destination)
    return destination
if DOWNLOAD_AUDIO or PROFILE == 'confirmation' or RUN_FITS_REFERENCE:
    base = 'https://zenkelab.org/datasets'
    with urllib.request.urlopen(base + '/md5sums.txt', timeout=120) as response:
        md5_text = response.read().decode()
    md5 = {row.split()[1].lstrip('*'): row.split()[0] for row in md5_text.splitlines() if len(row.split()) == 2}
    provenance = {'source': base, 'md5sums_text': md5_text, 'raw_sha256': {}}
    for dataset, splits in [('shd', ['train', 'test']), ('ssc', ['train', 'valid', 'test'])]:
        for split in splits:
            name = f'{dataset}_{split}.h5.gz'
            assert name in md5, f'Official checksum missing for {name}'
            archive = fetch(base + '/' + name, DATA / dataset / name, md5[name], 'md5')
            raw = archive.with_suffix('')
            # Hash the gzip content even for an existing raw file: it must match the verified archive.
            h = hashlib.sha256()
            with gzip.open(archive, 'rb') as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b''): h.update(block)
            if raw.exists():
                assert sha256(raw) == h.hexdigest(), f'Raw file differs from official archive: {raw}'
            else:
                temporary = raw.with_suffix('.h5.part')
                with gzip.open(archive, 'rb') as stream, temporary.open('wb') as target: shutil.copyfileobj(stream, target)
                temporary.replace(raw)
            provenance['raw_sha256'][str(raw.relative_to(DATA))] = h.hexdigest()
    (DATA / 'download_provenance.json').write_text(json.dumps(provenance, indent=2))
    print('Official audio verified:', provenance['raw_sha256'])
else:
    print('Audio download skipped: synthetic smoke profile.')

## Run the registered runner
Validation selects the earliest best epoch. The final test set is evaluated
only after training; zero-state/direct-only and globally deranged-state probes
reuse that checkpoint. No architecture tuning is performed on test results.
Restart this cell to resume at epoch boundaries. Changing source, configuration,
data or environment requires a **new output directory**, not overwriting results.
Confirmation is fixed at 64 bins, 700 channels, seeds 310–319, 133,631-parameter
ceiling, AdamW, 15 epochs and batch 256. FiTS's 250-bin/140-channel recipe is
intentionally **not** a matched control.

In [ ]:
# 5. Execute. Logs stream live; Ctrl+C preserves the last completed epoch.
command([sys.executable, REPO / 'gen5/examples/evidence1_audio.py', '--profile', PROFILE,
         '--output-dir', OUTPUT, '--data-root', DATA,
         '--device', 'cuda' if torch.cuda.is_available() else 'cpu'], cwd=REPO)

In [ ]:
# 6. Results, confidence intervals and honest decision status.
results = json.loads((OUTPUT / 'results.json').read_text())
records = pd.DataFrame(results['records'])
display(records[['dataset', 'seed', 'arm', 'accuracy', 'best_validation_accuracy', 'best_epoch', 'parameters', 'train_seconds']])
print(json.dumps(results['summary'], indent=2))
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, dataset in zip(axes, ['shd', 'ssc']):
    subset = records[records.dataset == dataset]
    subset.groupby('arm').accuracy.mean().reindex(ARMS).plot.bar(ax=ax)
    ax.set(title=dataset.upper() + (' — SYNTHETIC SMOKE' if PROFILE == 'smoke' else ' — test accuracy'), ylim=(0, 1), ylabel='Accuracy')
fig.tight_layout()
fig.savefig(OUTPUT / 'accuracy_overview.png', dpi=150)
plt.show()
print('Smoke cannot unlock a research gate. Clean-clone replication and a reviewed decision lock remain required.')

## Optional literature assets (code pinned, papers provenance-recorded)
Sources: [FiTS](https://github.com/kaistmm/FiTS),
[timing-only SHD](https://github.com/neural-reckoning/temporal-shd),
[BulletTrains](https://github.com/ToddMorrill/snn-bullet-trains),
[BrainTrace](https://github.com/chaobrain/braintrace),
[dual-memory pathways](https://github.com/sunpengfei1122/Dual_memory_pathways),
[fly LIF reference](https://github.com/philshiu/Drosophila_brain_model).
Downloading does not execute or endorse their mechanisms. Review upstream
licenses before integrating code. Closed dual-memory and sparse-scatter claims
are not reopened by these assets. Male-CNS/Eon/ConnectomeX are background only;
they are not interchangeable with the pinned Shiu/FlyWire reference here.

In [ ]:
# 7. Download pinned code. Paper URLs track latest revisions, not immutable versions.
REFERENCES = {
    'fits': ('kaistmm/FiTS', '8fe7e376bae13fe75f357dcfbff6adf9c45f1077'),
    'timing': ('neural-reckoning/temporal-shd', '65d8f4d9589d9e1f48c073f877ac44830afd61a2'),
    'bullettrains': ('ToddMorrill/snn-bullet-trains', 'a7717295b129d6afd45bb560e393893b0a344ad5'),
    'braintrace': ('chaobrain/braintrace', '365a88db39ca2cdbbe7291671d548347eeb25b86'),
    'dual_memory': ('sunpengfei1122/Dual_memory_pathways', '05cc0a3306b311e37ef9bc439ba7ac5186242672'),
    'fly': ('philshiu/Drosophila_brain_model', '91bdd1e7dcf193f3e7ca5a8933497fcef63b7960'),
}
UPSTREAM = {}
requested = set(REFERENCES) if DOWNLOAD_REFERENCES else set()
if RUN_FITS_REFERENCE: requested.add('fits')
if RUN_FLY_REFERENCE: requested.add('fly')
if DOWNLOAD_TIMING_DATA: requested.add('timing')
for name in sorted(requested):
    owner, revision = REFERENCES[name]
    UPSTREAM[name] = pinned_checkout('https://github.com/' + owner + '.git', revision, WORK / 'references' / name)
asset_manifest = {'repositories': {k: {'repository': REFERENCES[k][0], 'commit': REFERENCES[k][1]} for k in UPSTREAM}, 'papers': {}}
if DOWNLOAD_REFERENCES:
    for name, identifier in [('FiTS', '2605.13071'), ('BeyondRateCoding', '2507.16043'),
                             ('BulletTrains', '2603.13283'), ('DendriCL_preprint', '2607.02283'),
                             ('ConnectomeX_preprint', '2610.08418')]:
        url = 'https://arxiv.org/pdf/' + identifier
        paper = fetch(url, WORK / 'papers' / (name + '.pdf'))
        assert paper.read_bytes()[:5] == b'%PDF-', f'Not a PDF: {paper}'
        asset_manifest['papers'][name] = {'url': url, 'sha256': sha256(paper),
            'revision_pinned': False, 'provenance': 'observed download hash, not immutable revision'}
(WORK / 'reference_assets.json').write_text(json.dumps(asset_manifest, indent=2))
print('Reference assets:', asset_manifest)

In [ ]:
# 8. Timing-only SHD Zenodo download; no training here until E1 decision is locked.
if DOWNLOAD_TIMING_DATA:
    with urllib.request.urlopen('https://zenodo.org/api/records/16153275', timeout=120) as response:
        metadata = json.load(response)
    files = metadata['files']
    display(pd.DataFrame([{'name': f['key'], 'bytes': f['size'], 'checksum': f['checksum']} for f in files]))
    selected = files if TIMING_FILE_NAMES is None else [f for f in files if f['key'] in TIMING_FILE_NAMES]
    if TIMING_FILE_NAMES is not None:
        assert set(TIMING_FILE_NAMES) == {f['key'] for f in selected}, 'Unknown timing-data file name'
    size = sum(f['size'] for f in selected) / 1e9
    assert size <= TIMING_DOWNLOAD_LIMIT_GB, f'{size:.2f} GB exceeds configured budget; inspect filenames and change the limit.'
    for file in selected:
        assert Path(file['key']).name == file['key'], 'Unsafe remote filename'
        algorithm, checksum = file['checksum'].split(':', 1)
        assert algorithm in ('md5', 'sha256')
        fetch(file['links']['self'], DATA / 'timing_only' / file['key'], checksum, algorithm)
    (DATA / 'timing_only/zenodo_metadata.json').write_text(json.dumps(metadata, indent=2))
    print('Verified timing-only files; archive extraction/training intentionally not automatic.')
else:
    print('Timing-only dataset download disabled.')

In [ ]:
# 9. Optional FiTS native-recipe references (NOT matched Evidence-1 arms).
if RUN_FITS_REFERENCE:
    for dataset in ['shd', 'ssc']:
        command([sys.executable, REPO / 'gen5/tools/fits_validation_reference.py', '--upstream', UPSTREAM['fits'],
                 '--dataset', dataset, '--data-root', DATA, '--output-dir', WORK / 'fits_reference' / dataset,
                 '--seed', 310, '--epochs', 100], cwd=UPSTREAM['fits'])
        summary = json.loads((WORK / 'fits_reference' / dataset / 'validation_reference_summary.json').read_text())
        print(dataset, summary)
else:
    print('FiTS native-recipe reference disabled; no test-only selection is used.')

In [ ]:
# 10. Optional fly reference diagnostic with author-example comparison.
# It does not train A-SNN or change its wiring. The actual topology-prior pilot is gated.
if RUN_FLY_REFERENCE:
    command([sys.executable, REPO / 'gen5/tools/fly_reference.py', '--upstream', UPSTREAM['fly'],
             '--output-dir', WORK / ('fly_' + FLY_PROFILE), '--profile', FLY_PROFILE], cwd=UPSTREAM['fly'])
    display(pd.read_csv(WORK / ('fly_' + FLY_PROFILE) / 'firing_rates_hz.csv').head())
else:
    print('Whole-fly diagnostic disabled. Enable it only when you have CPU time and memory available.')

## Download the deliverables
The final ZIP includes checkpoints, per-seed records, selection histories,
protocol/data/environment hashes, plots, and any enabled reference-run outputs.
Raw audio, upstream checkouts, and paper PDFs are excluded to keep it portable.
Returned checkpoints are trusted only if created by you with this notebook.
Bring back the ZIP for interpretation; do not tune from the test accuracy table.
A final E1 decision still requires clean-clone repetition and human review.

Next queued branches: rate-matched temporal perturbations and sliding-window
readout; FiTS matched port only under a new protocol; conditional associative
scans; a small connectome-prior pilot with signed/degree-preserving rewiring
controls; frozen-backbone explicit-context masks. No new mechanism is opened
automatically by the smoke run or the reference downloads.

In [ ]:
# 11. Rebuild E1 bundle after plotting, verify every hash, export a combined archive.
from ammc_gen5.evidence1_audio import bundle, digest
bundle(OUTPUT)
manifest = json.loads((OUTPUT / 'manifest.json').read_text())['files']
for relative, record in manifest.items():
    file = OUTPUT / relative
    assert file.stat().st_size == record['bytes'] and digest(file) == record['sha256'], relative
import zipfile
archive = WORK / 'research_restart_results.zip'
paths = list(OUTPUT.rglob('*'))
for folder in [WORK / 'fits_reference', WORK / ('fly_' + FLY_PROFILE)]:
    if folder.exists(): paths.extend(folder.rglob('*'))
paths.append(WORK / 'reference_assets.json')
if (DATA / 'download_provenance.json').exists(): paths.append(DATA / 'download_provenance.json')
members = sorted({p for p in paths if p.is_file() and p.suffix not in ('.zip', '.tmp', '.part')})
combined_manifest = {str(p.relative_to(WORK)).replace(os.sep, '/'): {'sha256': sha256(p), 'bytes': p.stat().st_size} for p in members}
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in members: z.write(p, p.relative_to(WORK))
    z.writestr('return_manifest.json', json.dumps(combined_manifest, indent=2))
with zipfile.ZipFile(archive) as z:
    for relative, record in combined_manifest.items():
        assert hashlib.sha256(z.read(relative)).hexdigest() == record['sha256'], relative
print('Verified return bundle:', archive, 'SHA-256:', sha256(archive))
if IN_COLAB and DOWNLOAD_RESULT_ZIP:
    from google.colab import files
    files.download(str(archive))
else:
    from IPython.display import FileLink
    display(FileLink(str(archive)))
    print('In Colab: enable DOWNLOAD_RESULT_ZIP or download this file from the Files sidebar/Drive.')